In [ ]:
%matplotlib inline
import sys, os, importlib
import numpy as np
import torch
import torch.nn as nn
import h5py, json, math, time
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
CKPT_BASE = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models"
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
TARGET_ROLLOUT_DIR = PROJECT_ROOT / "rollouts" / "target_policy_50"
DIFFUSION_SAVE_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v0252_traj_mse"
ORACLE_JSON = PROJECT_ROOT / "results/2026-03-12/oracle_eval_all_checkpoints.json"
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

# Dims
STATE_DIM = 19
ACTION_DIM = 7
TRANSITION_DIM = 26
CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84

# Diffuser config (must match v0.2.5.2 training)
CHUNK_SIZE = 4
N_DIFFUSION_STEPS = 256
BASE_DIM = 32
DIM_MULTS = (1, 4, 8)
ACTION_WEIGHT = 5.0

# Generation config
NUM_SYNTHETIC = 50
T_GEN = 60
GAMMA = 1.0

# Guidance config
SIGMOID_SHARPNESS = 150.0

# ── 2D Grid Search: score_timestep x action_scale ──
SCORE_TIMESTEPS = [5, 10, 20, 35, 50]
ACTION_SCALES = [0.1, 0.3, 0.5, 0.8, 1.0, 1.5]

# Target policies (same 6 as v0.2.5.14d/14f)
TARGET_POLICIES = [
    {"name": "10demos_epoch10", "dir": "lift_diffusion_10demos/20260311115828", "ckpt": "models/model_epoch_10.pth"},
    {"name": "100demos_epoch20", "dir": "lift_diffusion_100demos/20260311135551", "ckpt": "models/model_epoch_20.pth"},
    {"name": "test_checkpoint", "dir": "test/20260309132349", "ckpt": "last.pth"},
    {"name": "10demos_epoch30", "dir": "lift_diffusion_10demos/20260311115828", "ckpt": "models/model_epoch_30.pth"},
    {"name": "50demos_epoch30", "dir": "lift_diffusion_50demos/20260311134204", "ckpt": "models/model_epoch_30.pth"},
    {"name": "200demos_epoch40", "dir": "lift_diffusion_200demos/20260311141036", "ckpt": "models/model_epoch_40.pth"},
]

N_GRID = len(SCORE_TIMESTEPS) * len(ACTION_SCALES)
N_TOTAL = N_GRID * len(TARGET_POLICIES)
print(f"Grid: {len(SCORE_TIMESTEPS)} timesteps x {len(ACTION_SCALES)} scales = {N_GRID} cells")
print(f"Total: {N_GRID} x {len(TARGET_POLICIES)} policies = {N_TOTAL} guided runs")
print(f"Timesteps: {SCORE_TIMESTEPS}")
print(f"Scales: {ACTION_SCALES}")

In [ ]:
# ── Reward + OPE functions ──
def hard_reward(cube_z):
    return (cube_z > LIFT_THRESHOLD).astype(np.float32)

def sigmoid_reward(cube_z, center=LIFT_THRESHOLD, sharpness=SIGMOID_SHARPNESS):
    return (1.0 / (1.0 + np.exp(-sharpness * (cube_z - center)))).astype(np.float32)

def compute_ope_hard(states, gamma=1.0):
    B, T, _ = states.shape
    cube_z = states[:, :, CUBE_Z_INDEX]
    rewards = hard_reward(cube_z)
    gammas = gamma ** np.arange(T)
    return (rewards * gammas[None, :]).sum(axis=1)

def compute_ope_sigmoid(states, gamma=1.0):
    B, T, _ = states.shape
    cube_z = states[:, :, CUBE_Z_INDEX]
    rewards = sigmoid_reward(cube_z)
    gammas = gamma ** np.arange(T)
    return (rewards * gammas[None, :]).sum(axis=1)

def compute_sr_hard(states):
    B = states.shape[0]
    cube_z = states[:, :, CUBE_Z_INDEX]
    return np.mean([np.any(cube_z[j] > LIFT_THRESHOLD) for j in range(B)])

print("Reward/OPE functions defined.")

In [ ]:
# ── Load oracle ──
with open(ORACLE_JSON, "r") as f:
    oracle_all = json.load(f)

oracle_sr_map = {}
for pol in TARGET_POLICIES:
    name = pol["name"]
    if name == "test_checkpoint":
        with open(CKPT_BASE / "test/20260309132349/oracle_50.json", "r") as f:
            test_oracle = json.load(f)
        oracle_sr_map[name] = float(test_oracle["mean_return"])
    else:
        oracle_sr_map[name] = float(oracle_all[name]["mean_return"])

print("Oracle SR values:")
for name, sr in oracle_sr_map.items():
    print(f"  {name:<22} {sr*100:.0f}%")

In [ ]:
# ── Load data + normalization + diffuser ──
all_states_list, all_actions_list = [], []
target_data = []
for path in sorted(TARGET_ROLLOUT_DIR.glob("rollout_*.h5"))[:50]:
    with h5py.File(path, "r") as f:
        latents = f["latents"][:]
        actions = f["actions"][:]
    states = (latents[:, -1, :] if latents.ndim == 3 else latents).astype(np.float32)
    actions = actions.astype(np.float32)
    target_data.append({"states": states, "actions": actions})
    all_states_list.append(states)
    all_actions_list.append(actions)
print(f"Loaded {len(target_data)} target rollouts")

with h5py.File(DEMO_HDF5, "r") as f:
    for dk in sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1])):
        demo = f[f"data/{dk}"]
        states = np.concatenate([demo["obs"][k][:].astype(np.float32) for k in OBS_KEYS], axis=-1)
        actions = demo["actions"][:].astype(np.float32)
        all_states_list.append(states)
        all_actions_list.append(actions)
print(f"Loaded 200 expert demos")

all_states = np.concatenate(all_states_list, axis=0)
all_actions = np.concatenate(all_actions_list, axis=0)
norm_mean = np.concatenate([all_states.mean(0), all_actions.mean(0)]).astype(np.float32)
norm_std = np.maximum(np.concatenate([all_states.std(0), all_actions.std(0)]), 1e-6).astype(np.float32)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t

initial_states_t = torch.tensor(
    np.array([ep["states"][0] for ep in target_data[:NUM_SYNTHETIC]]),
    dtype=torch.float32, device=device
)
print(f"Initial states: {initial_states_t.shape}")

# Load diffuser
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE, transition_dim=TRANSITION_DIM,
    dim=BASE_DIM, dim_mults=DIM_MULTS, attention=False,
).to(device)
diffusion_model = GaussianDiffusion(
    model=temporal_model, horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM, action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn, unnormalizer=unnormalize_fn,
    predict_epsilon=False, loss_type="l2",
    clip_denoised=False, action_weight=ACTION_WEIGHT,
).to(device)
ema = EMA(diffusion_model)
ema.ema_model.load_state_dict(
    torch.load(DIFFUSION_SAVE_DIR / "diffusion_model_ema.pt", map_location=device)
)
print(f"Loaded EMA diffuser from {DIFFUSION_SAVE_DIR}")

In [ ]:
# ── Trajectory generator (positive-only guidance) ──

def generate_trajectories(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim, chunk_size, t_gen, device,
    target_scorer=None,
    action_scale=0.0, normalize_grad=True,
):
    guided = (target_scorer is not None and action_scale > 0)
    B = initial_states.shape[0]
    td = state_dim + action_dim
    n_ts = diffusion_model.n_timesteps

    pad = torch.cat([initial_states, torch.zeros(B, action_dim, device=device)], 1)
    cond_init = normalize_fn(pad)[:, :state_dim]
    all_traj = torch.zeros(B, t_gen, td, device=device)
    conditions = {0: cond_init}
    total = 0

    while total < t_gen:
        x = torch.randn(B, chunk_size, td, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_d in reversed(range(n_ts)):
            t_t = torch.full((B,), t_d, device=device, dtype=torch.long)
            with torch.no_grad():
                mm, _, mlv = diffusion_model.p_mean_variance(x=x, t=t_t)
                ms = torch.exp(0.5 * mlv)

            if guided:
                mm = unnormalize_fn(mm)
                sc = mm[:, :, :state_dim]
                ac = mm[:, :, state_dim:]

                tg = target_scorer.grad_log_prob_chunk(sc, ac)
                if normalize_grad:
                    tg = tg / (tg.norm(dim=-1, keepdim=True) + 1e-6)

                guide = torch.zeros_like(mm)
                guide[:, :, state_dim:] = tg
                mm = mm + action_scale * guide

                mm = normalize_fn(mm)
                mm = apply_conditioning(mm, conditions, state_dim)
                mm = unnormalize_fn(mm)
                mm = normalize_fn(mm)

            noise = torch.randn_like(x)
            x = mm + (1 - (t_d == 0) * 1.0) * ms * noise
            x = apply_conditioning(x, conditions, state_dim)

        chunk_u = unnormalize_fn(x)
        n_store = min(chunk_size - 1, t_gen - total)
        all_traj[:, total:total+n_store] = chunk_u[:, 1:1+n_store]
        total += n_store

        if total < t_gen:
            last_state_norm = x[:, -1, :state_dim]
            conditions = {0: last_state_norm}

    return all_traj.cpu().numpy()

print("Generator ready.")

In [ ]:
# ── Pre-load all target policy algos (load each checkpoint once) ──
target_algos = {}
for pol in TARGET_POLICIES:
    name = pol["name"]
    run_dir = CKPT_BASE / pol["dir"]
    print(f"Loading {name}...", end=" ", flush=True)
    t0 = time.time()
    ckpt = load_checkpoint(run_dir, ckpt_path=Path(pol["ckpt"]))
    target_algos[name] = build_algo_from_checkpoint(ckpt, device=str(device))
    print(f"{time.time()-t0:.0f}s")
print(f"\nAll {len(target_algos)} policy algos loaded.")

In [ ]:
# ── Generate unguided baseline (once) ──
print("Generating unguided trajectories...")
np.random.seed(42)
torch.manual_seed(42)

t0 = time.time()
unguided_trajs = generate_trajectories(
    diffusion_model=ema.ema_model,
    initial_states=initial_states_t,
    normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
    state_dim=STATE_DIM, action_dim=ACTION_DIM,
    chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
)
unguided_time = time.time() - t0

unguided_states = unguided_trajs[:, :, :STATE_DIM]
unguided_sr_hard = compute_sr_hard(unguided_states)
unguided_returns_hard = compute_ope_hard(unguided_states, GAMMA)
unguided_returns_sigmoid = compute_ope_sigmoid(unguided_states, GAMMA)

print(f"Unguided: hard SR={unguided_sr_hard*100:.0f}%, "
      f"hard OPE={unguided_returns_hard.mean():.3f}, "
      f"sigmoid OPE={unguided_returns_sigmoid.mean():.3f}, "
      f"{unguided_time:.0f}s")

In [ ]:
# ── 2D Grid Sweep: score_timestep x action_scale x target_policy ──
sweep_results = {}  # key: (timestep, scale, policy_name)
timestep_sigmas = {}  # map timestep -> sigma
t0_all = time.time()
total_runs = len(SCORE_TIMESTEPS) * len(ACTION_SCALES) * len(TARGET_POLICIES)
run_count = 0

for st in SCORE_TIMESTEPS:
    print(f"\n{'='*80}")
    print(f"SCORE_TIMESTEP = {st}")
    print(f"{'='*80}")

    # Build scorers for this timestep (reuse pre-loaded algos)
    scorers = {}
    for name, algo in target_algos.items():
        scorers[name] = RobomimicDiffusionScorer(
            algo, device=str(device), score_timestep=st, obs_keys=OBS_KEYS
        )

    # Record sigma for this timestep
    sigma = scorers[TARGET_POLICIES[0]["name"]].sigma
    timestep_sigmas[st] = sigma
    print(f"  sigma = {sigma:.4f}")

    for scale in ACTION_SCALES:
        print(f"\n  --- action_scale = {scale} ---")

        for pol in TARGET_POLICIES:
            name = pol["name"]
            run_count += 1
            oracle_sr = oracle_sr_map[name]

            print(f"    [{run_count}/{total_runs}] {name} (oracle={oracle_sr*100:.0f}%)...", end=" ", flush=True)
            np.random.seed(42)
            torch.manual_seed(42)
            t0 = time.time()

            guided_trajs = generate_trajectories(
                diffusion_model=ema.ema_model, initial_states=initial_states_t,
                normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
                state_dim=STATE_DIM, action_dim=ACTION_DIM,
                chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
                target_scorer=scorers[name],
                action_scale=scale, normalize_grad=True,
            )
            gen_time = time.time() - t0

            guided_states = guided_trajs[:, :, :STATE_DIM]
            sr = compute_sr_hard(guided_states)
            ope_hard_returns = compute_ope_hard(guided_states, GAMMA)
            ope_sigm_returns = compute_ope_sigmoid(guided_states, GAMMA)
            ope_hard = float(ope_hard_returns.mean())
            ope_sigm = float(ope_sigm_returns.mean())

            sweep_results[(st, scale, name)] = {
                "oracle_sr": oracle_sr,
                "guided_sr": sr,
                "ope_hard": ope_hard,
                "ope_sigmoid": ope_sigm,
                "ope_hard_std": float(ope_hard_returns.std()),
                "ope_sigm_std": float(ope_sigm_returns.std()),
            }
            print(f"{gen_time:.0f}s  SR={sr*100:.0f}%  hard={ope_hard:.3f}  sigm={ope_sigm:.3f}")

total_time = time.time() - t0_all
print(f"\nTotal sweep: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"\nTimestep -> sigma mapping:")
for st, sigma in timestep_sigmas.items():
    print(f"  t={st:>2}: sigma={sigma:.4f}")

In [ ]:
# ── Compute metrics per (timestep, scale) cell ──
policy_names = [p["name"] for p in TARGET_POLICIES]
oracle_srs = np.array([oracle_sr_map[n] for n in policy_names])

def regret_at_k(oracle_vals, ope_vals, k):
    true_topk = np.argsort(oracle_vals)[-k:]
    est_topk = np.argsort(ope_vals)[-k:]
    return float(oracle_vals[true_topk].mean() - oracle_vals[est_topk].mean())

grid_metrics = {}  # key: (timestep, scale)
for st in SCORE_TIMESTEPS:
    for scale in ACTION_SCALES:
        opes_hard = np.array([sweep_results[(st, scale, n)]["ope_hard"] for n in policy_names])
        opes_sigm = np.array([sweep_results[(st, scale, n)]["ope_sigmoid"] for n in policy_names])
        guided_srs = np.array([sweep_results[(st, scale, n)]["guided_sr"] for n in policy_names])

        opes_hard_norm = opes_hard / T_GEN
        opes_sigm_norm = opes_sigm / T_GEN

        rho_h, p_h = stats.spearmanr(oracle_srs, opes_hard)
        rho_s, p_s = stats.spearmanr(oracle_srs, opes_sigm)

        rmse_h = np.sqrt(np.mean((oracle_srs - opes_hard_norm) ** 2))
        rmse_s = np.sqrt(np.mean((oracle_srs - opes_sigm_norm) ** 2))

        grid_metrics[(st, scale)] = {
            "sigma": timestep_sigmas[st],
            "rho_hard": rho_h, "p_hard": p_h,
            "rho_sigm": rho_s, "p_sigm": p_s,
            "rmse_hard": rmse_h, "rmse_sigm": rmse_s,
            "regret1_hard": regret_at_k(oracle_srs, opes_hard, 1),
            "regret3_hard": regret_at_k(oracle_srs, opes_hard, 3),
            "regret1_sigm": regret_at_k(oracle_srs, opes_sigm, 1),
            "regret3_sigm": regret_at_k(oracle_srs, opes_sigm, 3),
            "sr_range": (guided_srs.min(), guided_srs.max()),
        }

print("Metrics computed for all grid cells.")

In [ ]:
# ── Summary tables ──
print("=" * 120)
print("v0.2.5.14i: 2D GRID SEARCH (score_timestep x action_scale)")
print(f"6 policies, pos-only guidance, normalize_grad=True")
print("=" * 120)

# Rho hard heatmap table
print(f"\n--- Spearman rho (HARD reward) ---")
header = f"{'t \\ scale':<10}" + "".join(f"{s:>8}" for s in ACTION_SCALES)
print(header)
print("-" * len(header))
for st in SCORE_TIMESTEPS:
    row = f"{st:<10}"
    for scale in ACTION_SCALES:
        rho = grid_metrics[(st, scale)]["rho_hard"]
        row += f"{rho:>+8.3f}"
    print(row)

# Rho sigmoid heatmap table
print(f"\n--- Spearman rho (SIGMOID reward) ---")
header = f"{'t \\ scale':<10}" + "".join(f"{s:>8}" for s in ACTION_SCALES)
print(header)
print("-" * len(header))
for st in SCORE_TIMESTEPS:
    row = f"{st:<10}"
    for scale in ACTION_SCALES:
        rho = grid_metrics[(st, scale)]["rho_sigm"]
        row += f"{rho:>+8.3f}"
    print(row)

# RMSE hard heatmap table
print(f"\n--- RMSE (HARD reward) ---")
header = f"{'t \\ scale':<10}" + "".join(f"{s:>8}" for s in ACTION_SCALES)
print(header)
print("-" * len(header))
for st in SCORE_TIMESTEPS:
    row = f"{st:<10}"
    for scale in ACTION_SCALES:
        rmse = grid_metrics[(st, scale)]["rmse_hard"]
        row += f"{rmse:>8.4f}"
    print(row)

# Regret@1 hard heatmap table
print(f"\n--- Regret@1 (HARD reward) ---")
header = f"{'t \\ scale':<10}" + "".join(f"{s:>8}" for s in ACTION_SCALES)
print(header)
print("-" * len(header))
for st in SCORE_TIMESTEPS:
    row = f"{st:<10}"
    for scale in ACTION_SCALES:
        r1 = grid_metrics[(st, scale)]["regret1_hard"]
        row += f"{r1:>8.3f}"
    print(row)

# Best cells
best_hard = max(grid_metrics, key=lambda k: grid_metrics[k]["rho_hard"])
best_sigm = max(grid_metrics, key=lambda k: grid_metrics[k]["rho_sigm"])
print(f"\nBest rho_hard: t={best_hard[0]}, scale={best_hard[1]} -> {grid_metrics[best_hard]['rho_hard']:+.3f}")
print(f"Best rho_sigm: t={best_sigm[0]}, scale={best_sigm[1]} -> {grid_metrics[best_sigm]['rho_sigm']:+.3f}")

# Top-5 cells by rho_hard
print(f"\nTop-5 cells (rho_hard):")
sorted_cells = sorted(grid_metrics, key=lambda k: grid_metrics[k]["rho_hard"], reverse=True)
for i, k in enumerate(sorted_cells[:5]):
    m = grid_metrics[k]
    print(f"  {i+1}. t={k[0]:>2}, scale={k[1]:<4} -> rho_H={m['rho_hard']:+.3f}, rho_S={m['rho_sigm']:+.3f}, R@1_H={m['regret1_hard']:.3f}")

# Top-5 cells by rho_sigm
print(f"\nTop-5 cells (rho_sigm):")
sorted_cells_s = sorted(grid_metrics, key=lambda k: grid_metrics[k]["rho_sigm"], reverse=True)
for i, k in enumerate(sorted_cells_s[:5]):
    m = grid_metrics[k]
    print(f"  {i+1}. t={k[0]:>2}, scale={k[1]:<4} -> rho_S={m['rho_sigm']:+.3f}, rho_H={m['rho_hard']:+.3f}, R@1_S={m['regret1_sigm']:.3f}")

In [ ]:
# ── Figure 1: Heatmaps (rho_hard, rho_sigm, RMSE_hard, regret@1_hard) ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics_to_plot = [
    ("rho_hard", "Spearman rho (Hard)", "RdYlGn", None),
    ("rho_sigm", "Spearman rho (Sigmoid)", "RdYlGn", None),
    ("rmse_hard", "RMSE (Hard)", "RdYlGn_r", None),
    ("regret1_hard", "Regret@1 (Hard)", "RdYlGn_r", None),
]

for ax, (key, title, cmap, vlim) in zip(axes.flat, metrics_to_plot):
    data = np.array([[grid_metrics[(st, sc)][key] for sc in ACTION_SCALES] for st in SCORE_TIMESTEPS])
    im = ax.imshow(data, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(ACTION_SCALES)))
    ax.set_xticklabels([str(s) for s in ACTION_SCALES])
    ax.set_yticks(range(len(SCORE_TIMESTEPS)))
    ax.set_yticklabels([str(t) for t in SCORE_TIMESTEPS])
    ax.set_xlabel("action_scale")
    ax.set_ylabel("score_timestep")
    ax.set_title(title)
    # Annotate cells
    for i in range(len(SCORE_TIMESTEPS)):
        for j in range(len(ACTION_SCALES)):
            val = data[i, j]
            color = "white" if abs(val - data.mean()) > data.std() else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8, color=color)
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("v0.2.5.14i: 2D Grid Search (6 policies)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Rho vs action_scale, one line per timestep ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(SCORE_TIMESTEPS)))

for ax, rho_key, title in zip(axes, ["rho_hard", "rho_sigm"], ["Rho Hard", "Rho Sigmoid"]):
    for i, st in enumerate(SCORE_TIMESTEPS):
        rhos = [grid_metrics[(st, sc)][rho_key] for sc in ACTION_SCALES]
        ax.plot(ACTION_SCALES, rhos, 'o-', color=colors[i], lw=2, label=f"t={st}")
    ax.axhline(0.8, color="green", ls="--", alpha=0.5, label="Target (0.8)")
    ax.axhline(0.886, color="red", ls=":", alpha=0.5, label="Prior best (0.886)")
    ax.set_xlabel("action_scale")
    ax.set_ylabel("Spearman rho")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.2, 1.05)

fig.suptitle("v0.2.5.14i: Rho vs Action Scale by Timestep", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 3: Rho vs score_timestep, one line per action_scale ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(ACTION_SCALES)))

for ax, rho_key, title in zip(axes, ["rho_hard", "rho_sigm"], ["Rho Hard", "Rho Sigmoid"]):
    for i, sc in enumerate(ACTION_SCALES):
        rhos = [grid_metrics[(st, sc)][rho_key] for st in SCORE_TIMESTEPS]
        ax.plot(SCORE_TIMESTEPS, rhos, 'o-', color=colors[i], lw=2, label=f"scale={sc}")
    ax.axhline(0.8, color="green", ls="--", alpha=0.5, label="Target (0.8)")
    ax.axhline(0.886, color="red", ls=":", alpha=0.5, label="Prior best (0.886)")
    ax.set_xlabel("score_timestep")
    ax.set_ylabel("Spearman rho")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.2, 1.05)

fig.suptitle("v0.2.5.14i: Rho vs Timestep by Action Scale", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 4: OPE scatter at the best (timestep, scale) cell ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, best_key, rho_key, title in zip(
    axes,
    [best_hard, best_sigm],
    ["rho_hard", "rho_sigm"],
    ["Best cell (rho_hard)", "Best cell (rho_sigm)"],
):
    st, sc = best_key
    opes = np.array([sweep_results[(st, sc, n)]["ope_hard" if "hard" in rho_key else "ope_sigmoid"] / T_GEN for n in policy_names])
    rho = grid_metrics[(st, sc)][rho_key]

    ax.scatter(oracle_srs * 100, opes * 100, s=100, c="coral", edgecolor="black", zorder=5)
    for j, n in enumerate(policy_names):
        label = n.replace("demos_epoch", "e").replace("test_checkpoint", "test")
        ax.annotate(label, (oracle_srs[j]*100, opes[j]*100),
                    textcoords="offset points", xytext=(5, 5), fontsize=7)
    ax.plot([0, 100], [0, 100], 'k--', alpha=0.3)
    ax.set_xlabel("Oracle SR (%)")
    ax.set_ylabel("OPE (normalized %)")
    ax.set_title(f"{title}\nt={st}, scale={sc}, rho={rho:+.3f}")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5, 100)
    ax.set_ylim(-5, 100)

fig.suptitle("v0.2.5.14i: Oracle vs OPE at Best Grid Cells", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 5: Comparison to prior experiments ──
print("\n" + "=" * 80)
print("COMPARISON TO PRIOR EXPERIMENTS")
print("=" * 80)

best_h_m = grid_metrics[best_hard]
best_s_m = grid_metrics[best_sigm]

rows = [
    ("v0.2.5.14 (baseline)", 6, 0.714, 0.714, "scale=0.01, t=5"),
    ("v0.2.5.14d (scale sweep)", 6, 0.886, 0.943, "scale=0.4/0.3, t=5"),
    ("v0.2.5.14f (timestep sweep)", 6, 0.771, 0.886, "scale=0.01, t=50/20"),
    ("v0.2.5.14i (GRID SEARCH)", 6, best_h_m["rho_hard"], best_s_m["rho_sigm"],
     f"t={best_hard[0]}/scale={best_hard[1]} (H), t={best_sigm[0]}/scale={best_sigm[1]} (S)"),
]

print(f"\n{'Experiment':<35} {'Pols':>4} {'rho_H':>7} {'rho_S':>7} {'Best setting'}")
print("-" * 100)
for name, npol, rho_h, rho_s, setting in rows:
    marker = " <-- THIS" if "14i" in name else ""
    print(f"{name:<35} {npol:>4} {rho_h:>+7.3f} {rho_s:>+7.3f}  {setting}{marker}")

# Check if we beat prior best
if best_h_m["rho_hard"] > 0.886 or best_s_m["rho_sigm"] > 0.943:
    print("\nRESULT: NEW RECORD! Grid search found a better combination.")
elif best_h_m["rho_hard"] >= 0.886 and best_s_m["rho_sigm"] >= 0.886:
    print("\nRESULT: SUCCESS — matches or exceeds prior bests.")
else:
    print(f"\nRESULT: Best grid rho_H={best_h_m['rho_hard']:+.3f}, rho_S={best_s_m['rho_sigm']:+.3f}")

In [ ]:
# ── Summary text figure ──
fig, ax = plt.subplots(1, 1, figsize=(12, 14))
ax.axis("off")

lines = [
    "v0.2.5.14i: 2D Grid Search (score_timestep x action_scale)",
    "=" * 60,
    "",
    f"Diffuser: v0.2.5.2 EMA (200 demos + 50 rollouts)",
    f"Guidance: pos-only, normalize_grad=True",
    f"Grid: {SCORE_TIMESTEPS} x {ACTION_SCALES}",
    f"6 policies, {NUM_SYNTHETIC} trajs, T_GEN={T_GEN}",
    f"Total runs: {total_runs}, time: {total_time/60:.1f} min",
    "",
    "--- Spearman rho (Hard) ---",
    f"{'t':<6}" + "".join(f"{s:>7}" for s in ACTION_SCALES),
]
for st in SCORE_TIMESTEPS:
    row = f"{st:<6}"
    for scale in ACTION_SCALES:
        rho = grid_metrics[(st, scale)]["rho_hard"]
        row += f"{rho:>+7.3f}"
    lines.append(row)

lines += [
    "",
    "--- Spearman rho (Sigmoid) ---",
    f"{'t':<6}" + "".join(f"{s:>7}" for s in ACTION_SCALES),
]
for st in SCORE_TIMESTEPS:
    row = f"{st:<6}"
    for scale in ACTION_SCALES:
        rho = grid_metrics[(st, scale)]["rho_sigm"]
        row += f"{rho:>+7.3f}"
    lines.append(row)

lines += [
    "",
    f"Best rho_hard: t={best_hard[0]}, scale={best_hard[1]} -> {best_h_m['rho_hard']:+.3f}",
    f"Best rho_sigm: t={best_sigm[0]}, scale={best_sigm[1]} -> {best_s_m['rho_sigm']:+.3f}",
    "",
    f"Unguided: hard={float(unguided_returns_hard.mean()):.3f}, sigm={float(unguided_returns_sigmoid.mean()):.3f}",
]

best_rho = max(best_h_m["rho_hard"], best_s_m["rho_sigm"])
if best_rho > 0.9:
    lines.append("\nRESULT: STRONG SUCCESS (best rho > 0.9)")
elif best_rho > 0.8:
    lines.append(f"\nRESULT: SUCCESS (best rho = {best_rho:.3f})")
else:
    lines.append(f"\nRESULT: PARTIAL (best rho = {best_rho:.3f})")

ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes,
        fontsize=10, verticalalignment="top", fontfamily="monospace")
plt.tight_layout()
plt.show()